In [ ]:
# !poetry run jupyter lab --ServerApp.token='' --ServerApp.password=''


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
repo_root = pathlib.Path().resolve()  # adjust if you launched Jupyter in a subdir
# sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))



In [2]:

from pprint import pprint
from plainera_unacronym.nlp.execute import detect_and_extract
import json

# 2) Sanity test (text path)
text = """The European Medicines Agency (EMA) approves drugs in the EU. The decision has drawn fierce criticism from the Israeli government, families of hostages held in Gaza and some Conservatives. Responding on Sunday, Israeli Prime Minister Benjamin Netanyahu said a Palestinian state "will not happen". While an exponential moving average (EMA) is a trading indicator."""
json_str = detect_and_extract(text)
pprint(json_str)   # pretty JSON in the cell


(DetectorResult(unique_acronyms={'EMA': FirstOccurrence(acronym='EMA',
                                                        start_offset=31,
                                                        end_offset=34,
                                                        confidence=0.85,
                                                        normalized_key='EMA')},
                occurrences=[Occurrence(acronym='EMA',
                                        start_offset=31,
                                        end_offset=34,
                                        confidence=0.85,
                                        context_window=(0, 61),
                                        normalized_key='EMA',
                                        reasons=None),
                             Occurrence(acronym='EMA',
                                        start_offset=334,
                                        end_offset=337,
                                        confidence=0.85,
 

In [ ]:
import textwrap

LONG_TEXT = textwrap.dedent("""
    The American Psychological Association (APA) publishes influential journals, while the American Planning Association (APA) guides urban development.
    In finance, the Consumer Price Index (CPI) tracks inflation, whereas in computing CPI often refers to cycles per instruction.
    Single Sign-On (SSO) simplifies access, and SSO stands for Single Sign-On across most IT platforms. Lightweight Directory Access Protocol (LDAP) integrates
    with Transport Layer Security (TLS) for secure binds in many NHS trusts. Jacob says, ALRIGHTY THEN!

    The Americans with Disabilities Act (ADA) ensures accessibility, but the American Dental Association (ADA) sets clinical guidelines.
    Corporate Social Responsibility (CSR) shapes strategy, whereas a Certificate Signing Request (CSR) kicks off PKI workflows.
    We ran GPU–accelerated jobs and compared GPU results to CPU baselines; R&D will review them alongside I/O traces and S&P 500 sector notes.

    The European Medicines Agency (EMA) approves drugs in the EU, while an exponential moving average (EMA) is a trading indicator.
    Centers for Disease Control and Prevention (CDC) issue guidance; in data engineering, Change Data Capture (CDC) drives downstream updates.
    Return on Investment (ROI) guides budgets, but Region of Interest (ROI) guides image processing.

    The Department of Energy (DOE) funds basic research, while Design of Experiments (DOE) structures trials.
    The Securities and Exchange Commission (SEC) oversees markets; the Southeastern Conference (SEC) organizes college sports.
    The Central Processing Unit (CPU) remains a baseline as Graphics Processing Units (GPU) scale out; Random Access Memory (RAM) capacity still gates workloads.
    “IT was tricky to reproduce” is just a sentence start, but IT (Information Technology) owns the SSO/LDAP stack.

    Digital Subscriber Line (DSL) brought early broadband; a Domain-Specific Language (DSL) made our pipeline concise.
    A Peripheral Component Interconnect (PCI) slot differs from Payment Card Industry (PCI) compliance.
    Earnings Per Share (EPS) moved after ISO-certified audits; Encapsulated PostScript (EPS) assets rendered crisply.
    We exported JSON and CSV snapshots; H2O chemistry demos stayed separate from MP3 decoding tests.

    The International Telecommunication Union (ITU) sets standards; the International Triathlon Union (ITU) runs competitions.
    The National Archives and Records Administration (NARA) preserves documents; the North American Retail Association (NARA) advocates for merchants.
    O’RAN specs advanced; USB-C hubs shipped. OK, we’ll regroup at 10:45 AM, and PM (Project Manager) will chair; later, PM stands for particulate matter.

    The British Standards Institution (BSI) audits suppliers; Business Systems Integration (BSI) teams coordinate ERP rollouts.
    Quality Assurance (QA) wrote test plans; Quality Control (QC) validated outputs.
    User Experience (UX) and User Interface (UI) workshops ran back-to-back; Estimated Time of Arrival (ETA) for the next build is 18:30.
    Note that U.S. and U.K. dotted forms appear here but aren’t acronyms under our pattern.

    The World Wide Web Consortium (W3C) advanced specs; HyperText Markup Language (HTML) docs and Cascading Style Sheets (CSS) were version-locked.
    The International Organization for Standardization (ISO) reviewed findings; the Insurance Services Office (ISO) published actuarial updates.
    The Federal Communications Commission (FCC) ruled on spectrum; Field-Programmable Gate Arrays (FPGA) sped up TLS offload.

    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()

SHORT_TEXT = textwrap.dedent("""
    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()



In [ ]:
json_str = detect_and_extract(LONG_TEXT)
pprint(json_str)

In [ ]:
# tests/test_acronyms_rd.py

text = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads."
data = json.loads(run_detection(text, as_json=True))
keys = set(data["unique_acronyms"].keys())
print(keys)
assert "R&D" in keys
assert data["unique_acronyms"]["R&D"]["confidence"] >= 0.60


In [ ]:
# --- setup (adjust path if needed) ---
import sys, pathlib, json

try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

def detect(text: str, **kwargs) -> dict:
    """Call your detector and return a parsed dict."""
    return json.loads(run_detection(text, as_json=True, **kwargs))

In [ ]:



# --- tiny test harness ---
PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1

def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def counts_by_acronym(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out


# --- tests ---

def test_rd_detected_and_normalized():
    d = detect("We’ll loop in R & D after the NHS workshop.")
    assert "R&D" in keys(d), f"got {keys(d)}"
    assert d["unique_acronyms"]["R&D"]["confidence"] >= 0.60

def test_ok_and_am_drop_in_normal_prose():
    d = detect("OK, let's meet at 10:30 AM after lunch.")
    ks = keys(d)
    assert "OK" not in ks, f"OK leaked in: {ks}"
    assert "AM" not in ks, f"AM leaked in: {ks}"

def test_it_drops_as_pronoun_but_kept_with_definition():
    d1 = detect("IT was raining when we arrived.")
    assert "IT" not in keys(d1), f"pronoun IT should drop, got {keys(d1)}"

    d2 = detect("IT (Information Technology) owns the LDAP stack.")
    assert "IT" in keys(d2), f"IT with definition should keep, got {keys(d2)}"
    assert d2["unique_acronyms"]["IT"]["confidence"] >= 0.72

def test_stands_for_directional_rightward():
    d = detect("AM stands for amplitude modulation. OK, fine.")
    assert "AM" in keys(d), f"AM in definitional context should keep, got {keys(d)}"
    assert d["unique_acronyms"]["AM"]["confidence"] >= 0.72
    assert "OK" not in keys(d), f"OK should drop, got {keys(d)}"

def test_curly_apostrophe_and_hyphen_variants():
    d = detect("O’RAN and USB-C are on the agenda with the NHS.")
    ks = keys(d)
    assert "O'RAN" in ks, f"O'RAN missing, got {ks}"
    assert "USB-C" in ks, f"USB-C missing, got {ks}"
    assert "NHS" in ks

def test_mixed_alnum_kept():
    d = detect("H2O and MP3 appear in the doc.")
    ks = keys(d)
    assert "H2O" in ks, f"H2O missing, got {ks}"
    assert "MP3" in ks, f"MP3 missing, got {ks}"

def test_length_aware_threshold_blocks_bare_two_letter():
    d = detect("We will use AI and GPU for this.")
    ks = keys(d)
    assert "AI" not in ks, f"AI should drop under 2-letter threshold, got {ks}"
    assert "GPU" in ks, f"GPU should pass, got {ks}"

def test_company_suffixes_drop_without_definition():
    d = detect("Acme LTD and Example PLC signed the MOU.")
    ks = keys(d)
    assert "LTD" not in ks, f"LTD leaked in: {ks}"
    assert "PLC" not in ks, f"PLC leaked in: {ks}"
    # don't assert on MOU (might or might not be detected depending on your config)

def test_parallel_matches_serial_results():
    base = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
    big = base * 200  # large enough to trigger parallel path (if enabled)
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel), f"unique sets differ: {keys(serial)} vs {keys(parallel)}"
    assert counts_by_acronym(serial) == counts_by_acronym(parallel), \
        f"occurrence counts differ: {counts_by_acronym(serial)} vs {counts_by_acronym(parallel)}"


# --- run all ---
tests = [
    ("R&D detected & normalized", test_rd_detected_and_normalized),
    ("OK/AM drop in prose", test_ok_and_am_drop_in_normal_prose),
    ("IT: pronoun drops, definition keeps", test_it_drops_as_pronoun_but_kept_with_definition),
    ("'stands for' directional", test_stands_for_directional_rightward),
    ("Curly apostrophe & hyphen", test_curly_apostrophe_and_hyphen_variants),
    ("Mixed alnum kept", test_mixed_alnum_kept),
    ("Length-aware threshold on 2-letter", test_length_aware_threshold_blocks_bare_two_letter),
    ("Company suffixes drop", test_company_suffixes_drop_without_definition),
    ("Parallel parity", test_parallel_matches_serial_results),
]

for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
from typing import Any
# --- longer notebook tests (no pytest needed) ---
import sys, pathlib, json, random, textwrap, itertools as it

# ensure package importable in notebook
try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

def detect(text: str, **kwargs) -> Any:
    return detect_and_extract(text)

PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1

def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def count_by_key(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out


# 1) Long paragraph with multiple signals (defs, separators, mixed alnum, distractors)
def test_long_mixed_paragraph():
    text = textwrap.dedent("""
        At 10:30 AM we met the NHS analytics team. IT (Information Technology) owns LDAP and SSO.
        The R & D unit collaborates with the GPU cluster for H2O simulations and MP3 decoding benchmarks.
        OK, let's add O’RAN to the agenda alongside USB-C adapters. Later, AM stands for amplitude modulation in RF.
        We will avoid dotted initialisms like U.S. here and noisy tokens like C++ or A+B which are not acronyms.
    """).strip()
    d = detect(text)
    ks = keys(d)
    # keep
    assert "NHS" in ks
    assert "IT" in ks and d["unique_acronyms"]["IT"]["confidence"] >= 0.72
    assert "R&D" in ks  # normalized from "R & D"
    assert "GPU" in ks and "H2O" in ks and "MP3" in ks
    assert "O'RAN" in ks and "USB-C" in ks
    # drop
    assert "OK" not in ks           # interjection
    assert "AM" in ks               # kept because of definitional sentence later ("AM stands for ...")
    assert "US" not in ks           # dotted variant shouldn't be matched by your pattern
    # sanity: C++ and A+B not treated as acronyms
    assert all(acr not in {"C++", "A+B"} for acr in ks)

# 2) Very long distance between token and definition should NOT boost (directional + windowed)
def test_directional_window_limit():
    filler = " lorem ipsum dolor sit amet," * 20  # ~600+ chars
    text = f"IT {filler} stands for Information Technology."  # 'stands for' too far to the right
    d = detect(text)
    assert "IT" not in keys(d), "IT should not be boosted by far-away 'stands for'"

# 3) First-occurrence mapping remains stable with normalization
def test_first_occurrence_normalization_stable():
    text = "R & D met R&D after lunch. R & D then emailed."
    d = detect(text)
    assert "R&D" in keys(d)
    first = d["unique_acronyms"]["R&D"]
    # The first span should be the earliest occurrence in text
    assert first["start_offset"] == text.index("R & D")

# 4) Repetition & counts in a larger synthetic corpus
def test_large_repetition_counts():
    base = "NHS and R&D work with GPU and USB-C. "
    text = base * 250  # 250 repetitions
    d = detect(text)
    counts = count_by_key(d)
    # Expect ~250 occurrences each (some tokens might appear twice per base, adjust as needed)
    for acr in ["NHS", "R&D", "GPU", "USB-C"]:
        assert acr in counts and counts[acr] >= 230, f"{acr} count too low: {counts.get(acr)}"

# 5) Hyphen/en-dash noise: GPU should still be detected; right-hand lower token shouldn't be needed
def test_gpu_with_following_dash_word():
    text = "We tested GPU–accelerated pipelines and GPU-accelerated kernels yesterday."
    d = detect(text)
    ks = keys(d)
    assert "GPU" in ks, "GPU should be detected even when followed by dash-word"
    # Ensure we didn't create a weird token spanning the dash
    assert all(o["acronym"] != "GPU–accelerated" for o in d["occurrences"])

# 6) Company suffixes and short common uppers drop (unless defined)
def test_company_suffixes_drop_and_ok_drops():
    text = "Acme LTD acquired Example PLC. OK, moving on. DR Smith arrived at 7 AM."
    d = detect(text)
    ks = keys(d)
    assert "LTD" not in ks and "PLC" not in ks
    assert "OK" not in ks
    assert "AM" not in ks  # time-of-day
    # DR is in non_acronym_upper → drop unless explicit definition
    assert "DR" not in ks

# 7) Parenthetical definition rescues otherwise noisy tokens
def test_parenthetical_rescue_for_ok_and_short_tokens():
    text = "OK (Object Kernel) appeared in legacy docs; AI (Artificial Intelligence) and IT (Information Technology) led."
    d = detect(text)
    ks = keys(d)
    assert "OK" in ks and d["unique_acronyms"]["OK"]["confidence"] >= 0.72
    assert "AI" in ks and d["unique_acronyms"]["AI"]["confidence"] >= 0.72
    assert "IT" in ks

# 8) Parallel == serial parity on a long doc
def test_parallel_parity_long_doc():
    para = (
        "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
        "O’RAN and USB-C were discussed. GPU outperformed CPU. "
    )
    big = para * 300
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel)
    assert count_by_key(serial) == count_by_key(parallel)

# 9) Context window snaps to sentence boundaries (lenient check)
def test_context_window_sentence_bounds():
    text = "Alpha. We met the NHS team today and they agreed. Beta."
    d = detect(text)
    nhs_occ = next(o for o in d["occurrences"] if o["acronym"] == "NHS")
    left, right = nhs_occ["context_window"]
    segment = text[left:right]
    assert segment.strip().endswith("agreed."), f"window right seems off: {segment!r}"
    assert segment.strip().startswith("We met the"), f"window left seems off: {segment!r}"

# 10) Long definitional form lines (parentheses) still boost within limit
def test_long_parenthetical_still_boosts_within_limit():
    long_def = "(Information Technology and related shared infrastructure services)"
    text = f"IT {long_def} owns the platform."
    d = detect(text)
    assert "IT" in keys(d) and d["unique_acronyms"]["IT"]["confidence"] >= 0.72


# --- run all ---
tests = [
    ("Long mixed paragraph", test_long_mixed_paragraph),
    ("Directional window limit", test_directional_window_limit),
    ("First-occurrence normalization", test_first_occurrence_normalization_stable),
    ("Large repetition counts", test_large_repetition_counts),
    ("GPU with following dash word", test_gpu_with_following_dash_word),
    ("Company suffixes & OK drop", test_company_suffixes_drop_and_ok_drops),
    ("Parenthetical rescue for short tokens", test_parenthetical_rescue_for_ok_and_short_tokens),
    ("Parallel parity on long doc", test_parallel_parity_long_doc),
    ("Context window sentence bounds", test_context_window_sentence_bounds),
    ("Long parenthetical boost", test_long_parenthetical_still_boosts_within_limit),
]

PASSED = FAILED = 0
for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
# ---- big paragraphs: stress test in a single notebook cell ----
import sys, pathlib, json, textwrap

# Ensure package importable in the notebook
try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))



def detect(text: str, **kwargs) -> dict:
    return json.loads(run_detection(text, as_json=True, **kwargs))


def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())


def counts_by_key(d: dict) -> dict[str, int]:
    c = {}
    for o in d["occurrences"]:
        c[o["acronym"]] = c.get(o["acronym"], 0) + 1
    return c


# ---- long, mixed-content paragraphs ----
LONG_TEXT = textwrap.dedent("""
    The American Psychological Association (APA) publishes influential journals, while the American Planning Association (APA) guides urban development.
    In finance, the Consumer Price Index (CPI) tracks inflation, whereas in computing CPI often refers to cycles per instruction.
    Single Sign-On (SSO) simplifies access, and SSO stands for Single Sign-On across most IT platforms. Lightweight Directory Access Protocol (LDAP) integrates
    with Transport Layer Security (TLS) for secure binds in many NHS trusts. Jacob says, ALRIGHTY THEN!

    The Americans with Disabilities Act (ADA) ensures accessibility, but the American Dental Association (ADA) sets clinical guidelines.
    Corporate Social Responsibility (CSR) shapes strategy, whereas a Certificate Signing Request (CSR) kicks off PKI workflows.
    We ran GPU–accelerated jobs and compared GPU results to CPU baselines; R&D will review them alongside I/O traces and S&P 500 sector notes.

    The European Medicines Agency (EMA) approves drugs in the EU, while an exponential moving average (EMA) is a trading indicator.
    Centers for Disease Control and Prevention (CDC) issue guidance; in data engineering, Change Data Capture (CDC) drives downstream updates.
    Return on Investment (ROI) guides budgets, but Region of Interest (ROI) guides image processing.

    The Department of Energy (DOE) funds basic research, while Design of Experiments (DOE) structures trials.
    The Securities and Exchange Commission (SEC) oversees markets; the Southeastern Conference (SEC) organizes college sports.
    The Central Processing Unit (CPU) remains a baseline as Graphics Processing Units (GPU) scale out; Random Access Memory (RAM) capacity still gates workloads.
    “IT was tricky to reproduce” is just a sentence start, but IT (Information Technology) owns the SSO/LDAP stack.

    Digital Subscriber Line (DSL) brought early broadband; a Domain-Specific Language (DSL) made our pipeline concise.
    A Peripheral Component Interconnect (PCI) slot differs from Payment Card Industry (PCI) compliance.
    Earnings Per Share (EPS) moved after ISO-certified audits; Encapsulated PostScript (EPS) assets rendered crisply.
    We exported JSON and CSV snapshots; H2O chemistry demos stayed separate from MP3 decoding tests.

    The International Telecommunication Union (ITU) sets standards; the International Triathlon Union (ITU) runs competitions.
    The National Archives and Records Administration (NARA) preserves documents; the North American Retail Association (NARA) advocates for merchants.
    O’RAN specs advanced; USB-C hubs shipped. OK, we’ll regroup at 10:45 AM, and PM (Project Manager) will chair; later, PM stands for particulate matter.

    The British Standards Institution (BSI) audits suppliers; Business Systems Integration (BSI) teams coordinate ERP rollouts.
    Quality Assurance (QA) wrote test plans; Quality Control (QC) validated outputs.
    User Experience (UX) and User Interface (UI) workshops ran back-to-back; Estimated Time of Arrival (ETA) for the next build is 18:30.
    Note that U.S. and U.K. dotted forms appear here but aren’t acronyms under our pattern.

    The World Wide Web Consortium (W3C) advanced specs; HyperText Markup Language (HTML) docs and Cascading Style Sheets (CSS) were version-locked.
    The International Organization for Standardization (ISO) reviewed findings; the Insurance Services Office (ISO) published actuarial updates.
    The Federal Communications Commission (FCC) ruled on spectrum; Field-Programmable Gate Arrays (FPGA) sped up TLS offload.

    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()


# ---- notebook test harness (no pytest needed) ----
PASSED = FAILED = 0
def check(name, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1


# ---- tests ----

def test_expected_present_subset():
    d = detect(LONG_TEXT)
    ks = keys(d)
    must_have = {
        "APA","CPI","SSO","LDAP","TLS","NHS",
        "ADA","CSR","GPU","CPU","R&D","I/O","S&P",
        "EMA","CDC","ROI","DOE","SEC","RAM","IT",
        "DSL","PCI","EPS","JSON","CSV","H2O","MP3",
        "ITU","NARA","O'RAN","USB-C","PM",
        "BSI","QA","QC","UX","UI","ETA",
        "W3C","HTML","CSS","ISO","FCC","FPGA",
        "ICC","ECB","NATO","NOAA","NASA"
    }
    missing = sorted(must_have - ks)
    assert not missing, f"Missing expected acronyms: {missing}"

def test_expected_drops():
    d = detect(LONG_TEXT)
    ks = keys(d)
    assert "OK" not in ks, f"Interjection OK should drop, got {ks}"
    assert "AM" not in ks, f"Time-of-day AM should drop, got {ks}"
    assert "US" not in ks and "UK" not in ks, "Dotted U.S./U.K. should not match pattern"

def test_pm_rescued_by_definitions():
    d = detect(LONG_TEXT)
    pm = d["unique_acronyms"].get("PM")
    assert pm is not None, "PM should be kept due to '(Project Manager)' and 'stands for ...'"
    assert pm["confidence"] >= 0.72, f"PM confidence too low: {pm['confidence']}"

def test_it_kept_due_to_definition():
    d = detect(LONG_TEXT)
    it_ = d["unique_acronyms"].get("IT")
    assert it_ is not None and it_["confidence"] >= 0.72, f"IT not rescued by definition: {it_}"

def test_separators_normalized():
    d = detect(LONG_TEXT)
    ks = keys(d)
    for token in ("R&D","I/O","S&P","USB-C","O'RAN"):
        assert token in ks, f"{token} missing"
    assert "O’RAN" not in ks
    assert "O'RAN" in ks

def test_parallel_equals_serial_on_long_text():
    serial = detect(LONG_TEXT, parallel=False)
    parallel = detect(LONG_TEXT, parallel=True)
    assert keys(serial) == keys(parallel), "unique sets differ between serial and parallel"
    def counts(d):
        c = {}
        for o in d["occurrences"]:
            c[o["acronym"]] = c.get(o["acronym"], 0) + 1
        return c
    assert counts(serial) == counts(parallel), f"occurrence counts differ: {counts(serial)} vs {counts(parallel)}"


# ---- run tests ----
tests = [
    ("Expected present (subset)", test_expected_present_subset),
    ("Expected drops (OK/AM/U.S./U.K.)", test_expected_drops),
    ("PM rescued by definitions", test_pm_rescued_by_definitions),
    ("IT rescued by definition", test_it_kept_due_to_definition),
    ("Separators normalized", test_separators_normalized),
    ("Parallel == Serial", test_parallel_equals_serial_on_long_text),
]

PASSED = FAILED = 0
for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
def test_shouty_phrase_drops():
    text = "Jacob says, ALRIGHTY THEN! We’ll reconvene."
    d = detect(text)
    ks = keys(d)
    assert "ALRIGHTY" not in ks, f"shout interjection leaked: {ks}"
    assert "THEN" not in ks, f"shout tail leaked: {ks}"

check("Shouty ALL-CAPS phrase drops", test_shouty_phrase_drops)


In [ ]:
def test_shout_rule_does_not_kill_real_acronym():
    d = detect("Big news: NASA! launches today.")
    assert "NASA" in keys(d), "NASA wrongly dropped by shout rule"

check("Shout rule doesn’t kill real acronyms", test_shout_rule_does_not_kill_real_acronym)


In [ ]:
# ensure import path in your notebook, then:
import json

from plainera_unacronym.nlp.common.types import DetectorConfig, DetectorResult, ExtractionResult


def detect(text: str, cfg: DetectorConfig) -> dict:
    return json.loads(run_detection(text, as_json=True, parallel=False, pretty=False,
                                    # run_detection builds its own DetectorConfig; here we pass ours via Detector directly if you prefer
                                   ))

# OFF by default — dotted forms should NOT appear
cfg_off = DetectorConfig(enable_dotted=False)
txt = "The U.S. economy and U.K. policy differ. NASA leads."
from plainera_unacronym.nlp.detection.detector import Detector
d_off = Detector(cfg_off).detect(txt)
print("OFF keys:", list(d_off.unique_acronyms.keys()))
assert "US" not in d_off.unique_acronyms
assert "UK" not in d_off.unique_acronyms
assert "NASA" in d_off.unique_acronyms

# ON — dotted forms should be detected and normalized (dots removed)
cfg_on = DetectorConfig(enable_dotted=True, dotted_display='strip')
d_on = Detector(cfg_on).detect(txt)
print("ON  keys:", list(d_on.unique_acronyms.keys()))
assert "US" in d_on.unique_acronyms
assert "UK" in d_on.unique_acronyms

# Mixed dotted length
txt2 = "U.S.A. standards differ from U.S. rules."
d2 = Detector(cfg_on).detect(txt2)
ks2 = set(d2.unique_acronyms.keys())
print("Mixed keys:", ks2)
assert "USA" in ks2 and "US" in ks2


In [ ]:
import json, logging


logging.basicConfig(level=logging.INFO)
log = logging.getLogger("acronyms")

text = "IT (Information Technology) at NHS works with R&D and USB-C gear."
data = json.loads(run_detection(text, as_json=True, debug_reasons=True))

for occ in data["occurrences"]:
    log.info(
        "ACRONYM %-8s key=%-8s conf=%.2f span=(%d,%d) reasons=%s",
        occ["acronym"], occ["normalized_key"], occ["confidence"],
        occ["start_offset"], occ["end_offset"], ",".join(occ.get("reasons") or [])
    )


In [ ]:
import importlib

from plainera_unacronym.nlp.common.types import DetectorConfig, pattern_cache
from plainera_unacronym.nlp.detection.detector import Detector
import plainera_unacronym.nlp.detection.heuristics.core as core
importlib.reload(core)         # ensure we have the latest compile_pattern
pattern_cache.clear()          # avoid sta
# le compiled patterns

In [ ]:
cfg = DetectorConfig(enable_dotted=True)
pat = core.compile_pattern(cfg)
print("PATTERN:", pat.pattern)

assert "(?:[A-Z]\\.){2,}" in pat.pattern, "Dotted branch missing in pattern"
assert "(?:[A-Z][a-z]?){2,5}" in pat.pattern, "CamelCaps branch missing in pattern"

In [ ]:

txt = "The U.S. economy and U.K. policy differ. NASA leads."
cfg = DetectorConfig(enable_dotted=True, dotted_display='preserve')  # case-preserving for mixed-case is fine
d = Detector(cfg).detect(txt)
print(list(d.unique_acronyms.keys()))

In [ ]:
txt_dotted = "The U.S. economy and U.K policy differ. NASA leads."
dotted = Detector(DetectorConfig(enable_dotted=True)).detect(txt_dotted)
print("DOTTED keys:", list(dotted.unique_acronyms.keys()))
# If this fails, your normalize_key call isn't stripping dots.
assert "US" in dotted.unique_acronyms, dotted.unique_acronyms
assert "UK" in dotted.unique_acronyms, dotted.unique_acronyms

In [ ]:
txt_mixed = "Transport for London (TfL) runs the Tube. TfL operates buses."
mixed_on  = Detector(DetectorConfig(enable_mixed_case=True)).detect(txt_mixed)
mixed_off = Detector(DetectorConfig(enable_mixed_case=False)).detect(txt_mixed)
print("MIXED ON keys:", list(mixed_on.unique_acronyms.keys()))
print("MIXED OFF keys:", list(mixed_off.unique_acronyms.keys()))

# If this fails, either the camel branch isn't active or keys aren't uppercased.
assert "TfL" in mixed_on.unique_acronyms, mixed_on.unique_acronyms
assert "TFL" not in mixed_off.unique_acronyms, mixed_off.unique_acronyms

In [ ]:
from plainera_unacronym.nlp.detection.detector import Detector
from plainera_unacronym.nlp.common.types import DetectorConfig

def keys(d): return set(d.unique_acronyms.keys())

# 1) Separators + dotted
txt = 'R & D met R&D. USB-C and O’RAN followed. U.S. policy differs. NHS) ok.'
on  = Detector(DetectorConfig(enable_dotted=True)).detect(txt)
off = Detector(DetectorConfig(enable_dotted=False)).detect(txt)
assert "R&D" in keys(on) and "USB-C" in keys(on) and "O'RAN" in keys(on)
assert "US" in keys(on) and "US" not in keys(off)




In [ ]:
# 2) Mixed-case
txt2 = "Transport for London (TfL) runs the Tube. TfL operates buses."
mc_on  = Detector(DetectorConfig(enable_mixed_case=True)).detect(txt2)
mc_off = Detector(DetectorConfig(enable_mixed_case=False)).detect(txt2)
assert "TfL" in keys(mc_on)
assert "TFL" not in keys(mc_off)

In [ ]:
from plainera_unacronym.wiring.composition import sink
# 1) Ensure bio plugin registers with the global registry
import plainera_unacronym.core.domains.bio.plugin  # noqa: F401  <-- adjust path if yours differs

from dataclasses import replace
from plainera_unacronym.nlp.detection.domains.bio.plugin import BioPlugin
from plainera_unacronym.nlp.detection.detector import Detector
from plainera_unacronym.nlp.common.types import DetectorConfig

def keys(d):
    return set(d.unique_acronyms.keys())

def test_bio_plugin_detects_greek_utr_and_virus():
    txt = "Measured IFN-γ and mRNA in SARS-CoV-2 5′-UTR; IL-6 (95% CI 1.2–2.3). USA U.K."

    # Enable the bio domain (merge, don't replace existing domains)
    # cfg = replace(
    #     DetectorConfig(enable_dotted=True),
    #     enabled_domains=frozenset({"bio"}),
    #     domain_cfg={"bio": BioConfig()},
    # )

    det = Detector(cfg)
    res = det.detect(txt)
    ks = keys(res)
    print(ks)

    # Bio-y hits
    assert {"IFN-γ", "SARS-CoV-2", "IL-6"}.issubset(ks)
    # UTR may normalize primes; be flexible
    assert any("UTR" in k for k in ks)

# 2) Run it
test_bio_plugin_detects_greek_utr_and_virus()

In [ ]:
# Notebook cell 2
async def _ticker(duration=0.25, period=0.02):
    """Ticks while detection runs; if ticks>0 the loop wasn't blocked."""
    ticks = 0
    start = time.perf_counter()
    while time.perf_counter() - start < duration:
        await asyncio.sleep(period)
        ticks += 1
    return ticks


In [ ]:
import asyncio, time, threading
from dataclasses import dataclass
from plainera_unacronym.nlp.common.types import DetectorConfig
from plainera_unacronym.nlp.detection.detector import Detector
@dataclass
class DetectorResult:
    status: str
    worker_thread: str

cfg = replace(
    DetectorConfig(enable_dotted=True),
    enabled_domains=frozenset({"bio"}),
    domain_cfg={"bio": BioConfig()},
)

detector = Detector(cfg)

async def _ticker(duration=0.35, period=0.05):
    """Runs while detect_async is working to prove the loop isn't blocked."""
    ticks = 0
    start = time.perf_counter()
    while time.perf_counter() - start < duration:
        await asyncio.sleep(period)
        ticks += 1
    return ticks


In [ ]:
# Notebook cell 3
async def run_test():
    # Give it some work so timing/ticker are meaningful.
    base = "NASA will launch SLS with the ESA service module."
    text = " ".join([LONG_TEXT] * 30)  # enlarge input to simulate heavier work

    # --- Monkey-patch detect_parallel to record the worker thread name ---
    orig = detector.detect_parallel
    detector._last_worker_thread = None  # probe field

    def spy(text_arg):
        detector._last_worker_thread = threading.current_thread().name
        return orig(text_arg)

    detector.detect_parallel = spy  # patch on the instance

    try:
        # Run async wrapper + a ticker concurrently
        t0 = time.perf_counter()
        result_task = asyncio.create_task(detector.detect_async(text))
        ticks_task  = asyncio.create_task(_ticker())
        result, ticks = await asyncio.gather(result_task, ticks_task)
        elapsed = time.perf_counter() - t0

        # Also compute expected via the original sync function
        expected = orig(text)

        # ---- Assertions ----
        # 1) loop not blocked
        assert ticks > 0, f"event loop looked blocked (ticks={ticks})"

        # 2) results match (compare cheap invariants to avoid huge prints)
        assert len(result.occurrences) == len(expected.occurrences)
        assert len(result.unique_acronyms) == len(expected.unique_acronyms)

        # 3) ran off the loop thread (spy captured the worker thread)
        assert detector._last_worker_thread is not None
        assert detector._last_worker_thread != threading.current_thread().name, (
            f"detect_parallel ran on the loop thread: {detector._last_worker_thread}"
        )

        print(f"✅ non-blocking: {ticks} ticks while waiting (~{elapsed:.3f}s)")
        print(f"✅ results match: {len(result.occurrences)} occs / {len(result.unique_acronyms)} uniques")
        print(f"✅ ran in background thread: {detector._last_worker_thread}")

    finally:
        # restore the original method
        detector.detect_parallel = orig
        if hasattr(detector, "_last_worker_thread"):
            delattr(detector, "_last_worker_thread")

# In Jupyter you can await at top level:
await run_test()
